In [7]:
import pandas as pd

df = pd.read_csv("../Data/reviews_final.csv")
print(df.shape)

(34625, 12)


In [8]:
df["negative"] = (df["sentiment"] == "negative") * 100
df["recommend"] = df["do_recommend"] * 100

In [9]:
df["short_name"] = df["product_name"]
df["n_words"] = df["clean_text"].str.split().str.len()

print(df["category"].value_counts())

category
Tablets                  17596
Echo & Smart Speakers     7262
Fire TV                   5068
Kindle                    4137
Chargers, Cables           562
Name: count, dtype: int64


In [10]:
MIN_REVIEWS = 10

def product_table(category_df):
    table = category_df.groupby("short_name").agg(
        reviews=("rating", "size"),
        avg_rating=("rating", "mean"),
        pct_negative=("negative", "mean"),
        pct_recommend=("recommend", "mean"),
    ).round(2)
    table = table[table["reviews"] >= MIN_REVIEWS]
    return table.sort_values("avg_rating", ascending=False)

product_table(df[df["category"] == "Chargers, Cables"])

,reviews,avg_rating,pct_negative,pct_recommend
short_name,,,,
amazon echo fire tv power adapter,16,4.50,6.25,NaN
amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers,416,4.39,10.34,NaN
amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers,73,4.21,16.44,94.44
amazon fire hd 6 standing protective case 4 th generation 2014 release cayenne red,13,3.92,23.08,NaN
kindled x leather cover black fits 97 display latest 2 nd generation kindled xs,10,3.50,30.00,NaN
new amazon kindle fire hd 9 w power fast adapter charger micro usb angle cable,13,2.46,61.54,NaN


In [11]:
def complaints(category_df, product, n=2):
    unhappy = category_df[(category_df["short_name"] == product) & (category_df["rating"] <= 3) & (category_df["n_words"] >= 10)]
    examples = unhappy["clean_text"].sample(min(n, len(unhappy)), random_state=42)
    return [text[:150] for text in examples]

def pick_products(table):
    if len(table) < 2:
        return list(table.index[:3]), None
    worst = table["avg_rating"].idxmin()
    top = table.drop(worst).sort_values("reviews", ascending=False).index[:3]
    return list(top), worst

In [ ]:
def product_line(table, product):
    row = table.loc[product]
    line = f"{product}: {row['avg_rating']}/5 from {row['reviews']:.0f} reviews, {row['pct_negative']}% negative"
    if pd.notna(row["pct_recommend"]):
        line += f", {row['pct_recommend']}% recommend it"
    return line

def facts_text(category):
    category_df = df[df["category"] == category]
    table = product_table(category_df)
    top, worst = pick_products(table)

    lines = [f"Category: {category} ({len(category_df)} reviews)", "", "TOP PRODUCTS (most reviewed first):"]
    for product in top:
        lines.append("- " + product_line(table, product))
        for text in complaints(category_df, product):
            lines.append(f'    complaint: "{text}"')
    lines.append("")
    if worst is None:
        lines.append("WORST PRODUCT: none")
    else:
        lines.append("WORST PRODUCT: " + product_line(table, worst))
        for text in complaints(category_df, worst):
            lines.append(f'    complaint: "{text}"')
    return "\n".join(lines)

categories = sorted(df["category"].unique())
print(categories)
print(facts_text("Chargers, Cables"))

['Chargers, Cables', 'Echo & Smart Speakers', 'Fire TV', 'Kindle', 'Tablets']
Category: Chargers, Cables (562 reviews)

TOP PRODUCTS (most reviewed first):
- amazon 5 w usb official oem charger power adapter for fire tablets kind lee readers: 4.39/5 from 416 reviews, 10.34% negative
    complaint: "a waste of money . a waste of money. you don't need this. just plug your tablet into your computer and the charge lasts for a month"
    complaint: "only lasts for less than 1 year! . this is the third power charger that i have had to buy at cost of over 30 each since my kindle fire purchase. why c"
- amazon 9 w power fast official oem usb charger power adapter for fire tablets kind lee readers: 4.21/5 from 73 reviews, 16.44% negative, 94.44% recommend it
    complaint: "defective. . worked wonderful and fast the first couple of weeks and then just stopped working. i understand that sometimes things are just a defectiv"
    complaint: "power fast how about power not! . this doesn't power any

In [13]:
import os
import re
from openai import OpenAI

# ---- API key ----
api_key = os.environ.get("NVIDIA_API_KEY")
try:
    import streamlit as st
    api_key = st.secrets.get("NVIDIA_API_KEY", api_key)
except Exception:
    pass

client = OpenAI(base_url="https://integrate.api.nvidia.com/v1", api_key=api_key)

# ---- Model ----
MODEL = "nvidia/nemotron-3-super-120b-a12b"


def clean(text):
    text = re.sub(r"<think>.*?</think>", "", text or "", flags=re.S)
    return text.strip()


def write(prompt, system="You are a helpful assistant.", max_tokens=600, creative=False, think=True):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "system", "content": system},
                  {"role": "user", "content": prompt}],
        max_tokens=max_tokens + (3000 if think else 0),      # room for the thinking
        temperature=0.7 if creative else 0.2,
        top_p=0.9,
        extra_body={"chat_template_kwargs": {"enable_thinking": think}},
    )
    msg = response.choices[0].message
    return clean(msg.content)          # the API usually puts thinking in a separate field

In [14]:
print(write("Write a short blog article (about 150 words) recommending the top 3 kitchen appliances: "
            "Blender X (4.5 stars), Toaster Y (4.2), Kettle Z (4.0). One strength and one complaint each.",
            system="You are a product review writer.", creative=True))

**Top 3 Kitchen Appliances Worth Your Counter Space**

If you’re looking to upgrade your kitchen without breaking the bank, these three highly‑rated gadgets deliver solid performance and everyday convenience.

**Blender X – 4.5★**  
*Strength:* Its powerful 1,200‑watt motor crushes ice, frozen fruit and tough greens in seconds, giving you silky smoothies and soups with minimal effort.  
*Complaint:* The lid can be a bit tricky to lock securely, occasionally leading to a small splash when blending thick mixtures.

**Toaster Y – 4.2★**  
*Strength:* Wide slots and adjustable browning settings let you toast everything from artisanal bagels to thin slices of bread to perfection, with a quick‑release crumb tray for easy cleanup.  
*Complaint:* The heating elements take a moment to reach peak temperature, so the first round of toast may be slightly lighter than subsequent batches.

**Kettle Z – 4.0★**  
*Strength:* Rapid‑boil technology brings a full liter of water to a rolling boil in under

In [15]:
def prompt_v1(category):
    return "Write a blog post about these products.\n\n" + facts_text(category)

def prompt_v2(category):
    category_df = df[df["category"] == category]
    top, worst = pick_products(product_table(category_df))

    template = [f"# The best and worst {category.lower()}", "[intro of two sentences]", "", "## Top picks"]
    for product in top:
        template.append(f"**{product}**: [one or two sentences: what buyers like, and how it compares with the others]")
    template += ["", "## Common complaints", "[two or three sentences: the problems unhappy buyers mention]", ""]
    if worst is not None:
        template += [f"## Product to avoid: {worst}", "[one or two sentences: why it is the weakest]", ""]
    template += ["## Verdict", "[one sentence]"]

    return (facts_text(category) + "\n\n"
            "Fill in this article template. Replace every [...] with your own text and keep everything else as it is.\n"
            "RULES: Use only the facts above. Do not invent prices, features or numbers. "
            "Do not copy the complaints, say them in your own words. "
            "If the product to avoid has a rating of 4 or more, say it is the weakest of a good group, not a bad product.\n\n"
            + "\n".join(template))

In [16]:
def write_article(prompt):
    return write(
        prompt,
        system="You are a writer for a product review blog.",
        max_tokens=600,
        creative=False,      # low temperature, like do_sample=False before
    )

In [17]:
import re

def check_article(article, category):
    category_df = df[df["category"] == category]
    top, worst = pick_products(product_table(category_df))
    names = top + ([worst] if worst is not None else [])
    sections = ["top picks", "common complaints", "verdict"]
    numbers_in_facts = set(re.findall(r"\d+(?:\.\d+)?", facts_text(category)))
    numbers_in_article = set(re.findall(r"\d+(?:\.\d+)?", article.replace(",", "")))
    return {
        "words": len(article.split()),
        "sections": f"{sum(section in article.lower() for section in sections)}/{len(sections)}",
        "product names": f"{sum(name in article for name in names)}/{len(names)}",
        "invented numbers": sorted(numbers_in_article - numbers_in_facts),
    }

In [18]:
for name, prompt_function in {"v1": prompt_v1, "v2": prompt_v2}.items():
    article = write_article(prompt_function("Kindle"))
    print(f"\n========== {name}", check_article(article, "Kindle"))
    print(article)


========== v1 {'words': 1067, 'sections': '1/3', 'product names': '0/4', 'invented numbers': ['3', '4000', '7', '80']}
**Kindle Showdown: Which Amazon E‑Reader Deserves a Spot on Your Shelf?**  
*An in‑depth look at the top‑selling Kindles (and the one that fell short) based on over 4,000 real‑world reviews.*

---

### Why the Kindle Line Still Matters

Even as tablets and smartphones dominate our screens, the Kindle family remains the go‑to choice for serious readers. Its e‑ink displays are easy on the eyes, battery life stretches for weeks, and the ecosystem—Kindle Store, Audible integration, and Whispersync—keeps your library wherever you go.  

With **4,137 reviews** across the lineup, we can see which models truly deliver on that promise and where users are still left wanting more.

---

## The Leaders: Paperwhite, Voyage & Oasis

| Model | Avg. Rating | # Reviews | % Negative | % Recommend | Key Praise | Common Complaints |
|-------|-------------|-----------|------------|-------

In [22]:
articles = []
for category in categories:
    article = write_article(prompt_v2(category))
    print(category, check_article(article, category))
    articles.append(article)

with open("../Reports/articles.md", "w",encoding="utf-8") as file:
    file.write("\n\n---\n\n".join(articles))
print("saved", len(articles), "articles")

Chargers, Cables {'words': 406, 'sections': '3/3', 'product names': '4/4', 'invented numbers': ['60']}
Echo & Smart Speakers {'words': 214, 'sections': '3/3', 'product names': '2/2', 'invented numbers': ['3.8', '6600', '93', '96']}
Fire TV {'words': 171, 'sections': '3/3', 'product names': '1/1', 'invented numbers': []}
Kindle {'words': 426, 'sections': '3/3', 'product names': '4/4', 'invented numbers': ['3100', '94.0', '97.8', '98.2']}
Tablets {'words': 335, 'sections': '3/3', 'product names': '4/4', 'invented numbers': []}
saved 5 articles


In [26]:
import os
os.makedirs("../CSVs", exist_ok=True)

# Keep only the columns product_table / facts_text / pick_products use.
# Add the review text column if facts_text uses review texts (e.g. "text").
cols = ['id', 'asins', 'product_name', 'title', 'text', 'rating', 
        'do_recommend', 'clean_text', 'sentiment', 'meta_category', 
        'cluster', 'category', 'negative', 'recommend', 'short_name', 'n_words']
df[cols].to_csv("../CSVs/reviews_slim.csv.gz", index=False, compression="gzip")